Merscope output documentation
https://vizgen.com/wp-content/uploads/2023/06/91600001_MERSCOPE-Instrument-User-Guide_Rev-G.pdf

The cell_boundaries.parquet file
contains the boundaries of cells in microns, formatted as a data table using GeoPandas

note: vizgen .parquet file EntityID is int64, very large.

**This script process vizgen .parquet and barcode files to generate cell x gene expression file**

input: .parquet file using MULTIPOLYGON for cell segmentation, barcode file

output: geneExp.tsv

In [12]:
import pandas as pd
import geopandas as gpd
import numpy
import cv2
import skimage.io

In [13]:
parquet_file = "micron_space.parquet"
barcode_file = "barcodes.csv"
exp_file = "geneExp.tsv"

In [14]:
gdf_mosaic = gpd.read_parquet(parquet_file)
gdf_mosaic

,ID,EntityID,ZIndex,Geometry,Type,ZLevel,Name,ParentID,ParentType
0,0,1578230100020100001,0,"MULTIPOLYGON (((2525.809 -13.134, 2525.763 -12...",cell,1.5,None,None,None
1,1,1578230100020100003,0,"MULTIPOLYGON (((2572.566 -19.888, 2573.351 -17...",cell,1.5,None,None,None
4,2,1578230100020100007,0,"MULTIPOLYGON (((2506.251 -19.3, 2507.729 -19.5...",cell,1.5,None,None,None
5,3,1578230100020100008,0,"MULTIPOLYGON (((2544.871 -21.159, 2545.954 -20...",cell,1.5,None,None,None
6,4,1578230100020100009,0,"MULTIPOLYGON (((2499.506 -17.708, 2500.288 -15...",cell,1.5,None,None,None
...,...,...,...,...,...,...,...,...,...
437499,387412,1578230103360100035,6,"MULTIPOLYGON (((6963.151 7293.093, 6962.906 72...",cell,10.5,None,None,None
437500,387413,1578230103360100029,6,"MULTIPOLYGON (((6976.935 7297.19, 6977.165 729...",cell,10.5,None,None,None
437501,387414,1578230103360100036,6,"MULTIPOLYGON (((6919.335 7295.134, 6919.906 72...",cell,10.5,None,None,None
437502,387415,1578230103361100003,0,"MULTIPOLYGON (((6996.766 7280.56, 6997.744 728...",cell,1.5,None,None,None


In [15]:
minx, miny, maxx, maxy = gdf_mosaic.total_bounds
scale = 10
minx, miny, maxx, maxy

(-127.85738601277501,
 -110.64471669282081,
 7397.479206040777,
 7420.565629188312)

In [16]:
barcode = pd.read_csv(barcode_file)
barcode

,Unnamed: 0,barcode_id,global_x,global_y,global_z,x,y,fov,gene,transcript_id
0,170,53,6429.86100,83.253380,0.0,1444.00000,1812.35250,0,TRPV4,ENST00000261740
1,174,162,6355.01660,43.255306,0.0,751.00000,1442.00000,0,EYA2,ENST00000327619
2,147,168,6319.18950,30.542408,0.0,419.26465,1324.28800,0,ZKSCAN8,ENST00000330236
3,115,362,6449.94870,1.651576,0.0,1630.00000,1056.78030,0,SP1,ENST00000426431
4,37,445,6399.94500,-54.120150,0.0,1167.00000,540.37540,0,CACNA1H,ENST00000565831
...,...,...,...,...,...,...,...,...,...,...
18280736,84,490,714.74396,7359.526400,6.0,1062.00000,1471.76070,1316,Blank-6,Blank-6
18280737,188,498,817.01996,7405.776400,6.0,2009.00000,1900.00000,1316,Blank-14,Blank-14
18280738,35,505,805.27094,7274.377000,6.0,1900.21230,683.33905,1316,Blank-21,Blank-21
18280739,306,543,745.41595,7391.338400,6.0,1346.00000,1766.31860,1316,Blank-59,Blank-59


vizgen .parquet file EntityID is int64, very large. We need to map it to sequence intergers for memory size management

In [20]:
cells = gdf_mosaic['EntityID'].unique()
Ncell = gdf_mosaic['EntityID'].nunique()
EntityID_to_cellID = {key: value for key, value in zip(cells, list(range(Ncell)))}
cellID_to_EntityID = {key: value for key, value in zip(list(range(Ncell)), cells)}

In [6]:
barcode['global_z'].unique()

array([0., 1., 2., 3., 4., 5., 6.])

In [14]:
numpy.min(barcode['global_x']), numpy.max(barcode['global_x']), numpy.min(barcode['global_y']), numpy.max(barcode['global_y']) 

(-117.35385, 7380.419, -95.918465, 7410.92)

# multi z level masks as filled masks using value of cell identity

In [85]:
z_levels = gdf_mosaic["ZIndex"].value_counts().sort_index()
z_levels

ZIndex
0    52485
1    53289
2    56503
3    58648
4    58316
5    55391
6    52771
Name: count, dtype: int64

In [18]:
height = int((maxy - miny) * scale)
width = int((maxx - minx) * scale)

mask_stack =[]

for z in z_levels.index:
    df = gdf_mosaic[gdf_mosaic['ZIndex'] == z]

    # Initialize a blank image (mask)
    mask = numpy.zeros((height, width), dtype=numpy.int32)
    
    # Draw the polygons on the mask (filled mask, or outline)
    for idx, row in df.iterrows():
        geometry = row['Geometry']  # Get the geometry (Polygon/MultiPolygon)
        fill_value = EntityID_to_cellID[row['EntityID']]

        for polygon in geometry.geoms:
            exterior_coords = numpy.array([[int((x - minx) * scale), int((y-miny)*scale)] for x, y in polygon.exterior.coords], dtype=numpy.int32)
            cv2.fillPoly(mask, [exterior_coords], fill_value)

    mask_stack.append(mask)

# generate gene expression using barcode and masks

In [23]:
Ngene = barcode['gene'].nunique()
Ncell = gdf_mosaic['EntityID'].nunique()
expMatrix = pd.DataFrame(numpy.zeros((Ngene, Ncell)))
expMatrix.index = barcode['gene'].unique()
expMatrix.columns = gdf_mosaic['EntityID'].unique()
expMatrix

,1578230100020100001,1578230100020100003,1578230100020100007,1578230100020100008,1578230100020100009,1578230100020100010,1578230100020100011,1578230100020100012,1578230100020100013,1578230100020100014,...,1578230103360100027,1578230103360100028,1578230103360100029,1578230103360100030,1578230103360100031,1578230103360100034,1578230103360100035,1578230103360100036,1578230103360100037,1578230103361100003
TRPV4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
EYA2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZKSCAN8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SP1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CACNA1H,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FMO1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Blank-62,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P2RY12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
RMST,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
for idx, row in barcode.iterrows():
    z = int(row["global_z"])
    x = int((row["global_x"] - minx) * scale)
    y = int((row["global_y"] -miny) * scale)
    cell_id = mask_stack[z][x,y]
    if idx % 10000 ==0:
        print (idx)
    if cell_id !=0:
        gene = row['gene']
        cell_EntityID = cellID_to_EntityID[cell_id]
        expMatrix.loc[gene, cell_EntityID] = expMatrix.loc[gene, cell_EntityID]+1

In [25]:
expMatrix

,1578230100020100001,1578230100020100003,1578230100020100007,1578230100020100008,1578230100020100009,1578230100020100010,1578230100020100011,1578230100020100012,1578230100020100013,1578230100020100014,...,1578230103360100027,1578230103360100028,1578230103360100029,1578230103360100030,1578230103360100031,1578230103360100034,1578230103360100035,1578230103360100036,1578230103360100037,1578230103361100003
TRPV4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
EYA2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZKSCAN8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SP1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CACNA1H,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FMO1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Blank-62,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
P2RY12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
RMST,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
numpy.max(expMatrix)

146.0

In [28]:
expMatrix.to_csv(exp_file, sep = '\t')